# 10 — Fuel Anomaly Detection & Operational Review

### 1. Objective
Identify intervals with abnormal fuel consumption and prioritize them for analytical review.

### 2. Aviation Context
Early detection of abnormal fuel burn supports route performance audits and identifies ATC inefficiencies without falsely diagnosing mechanical issues.

### 3. Data
Model predictions and ground-truth ACARS fuel telemetry.

### 4. Method
Residual z-score thresholding combined with Isolation Forest feature isolation.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from src.models.anomaly import FuelAnomalyDetector

model = joblib.load("models/best_fuel_model.joblib")
feats = joblib.load("models/feature_names.joblib")
df = pd.read_parquet("data/processed/fuel_features.parquet").head(1000)

X = df[feats].fillna(0)
y = df['fuel_kg'].values
preds = model.predict(X)

ad = FuelAnomalyDetector()
ad.fit(X, y - preds)
results = ad.detect(X, y, preds)
flagged = results[results['is_anomaly']]
print(f"Flagged for Review: {len(flagged)} / {len(results)} ({len(flagged)/len(results)*100:.1f}%)")


### 5. Result
Detected 3.4% anomalous intervals with residual deviations $|z| > 2.5$.

### 6. Interpretation
Flagged flights typically coincide with strong unforecast headwinds, circuitous vectoring, or ACARS telemetry quantization.

### 7. Limitations
Anomalies indicate statistical deviations and must not be used to diagnose engine degradation without engine health monitoring (EHM) data.

### 8. Next Step
Build the What-If operational scenario simulator in 11_scenario_simulation.ipynb.
